In [21]:
# pip install rdkit-pypi torch torchani (if using ANI)
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
import numpy as np

SMILES = "N[C@@H](C)C(=O)O"  # L-alanine (neutral example). For zwitterion, protonate/deprotonate appropriately.
NAME = "alanine_neutral"
NCONF = 150                 # more for larger side chains
RMS_PRUNE = 0.4             # Å
KEEP_MMFF = 50              # keep this many lowest by MMFF energy
USE_ANI = True             # set True to compute ANI-2x energies here

def enumerate_tautomers(mol):
    te = rdMolStandardize.TautomerEnumerator()
    tforms = te.Enumerate(mol)
    # Deduplicate by canonical SMILES
    uniq = {}
    for m in tforms:
        cs = Chem.MolToSmiles(m, isomericSmiles=True)
        uniq.setdefault(cs, m)
    return list(uniq.values())

def prepare_mol(smiles):
    m = Chem.MolFromSmiles(smiles)
    m = Chem.AddHs(m)
    return m

def embed_minimize_confs(mol, nconf=NCONF):
    params = AllChem.ETKDGv3()
    params.pruneRmsThresh = -1.0   # we’ll prune ourselves
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=nconf, params=params)
    # MMFF minimize
    props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant='MMFF94s')
    e_list = []
    for cid in conf_ids:
        try:
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            ff.Minimize(maxIts=500)
            e = ff.CalcEnergy()
        except Exception:
            e = 1e9
        e_list.append((cid, e))
    return e_list

def prune_by_rmsd(mol, conf_ids, rms_cut=RMS_PRUNE):
    kept = []
    for cid in conf_ids:
        keep = True
        for kc in kept:
            rms = rdMolAlign.GetBestRMS(mol, mol, prbId=cid, refId=kc)
            if rms < rms_cut:
                keep = False
                break
        if keep:
            kept.append(cid)
    return kept

def mmff_rank_and_prune(mol, conf_energy_pairs, keep=KEEP_MMFF):
    conf_energy_pairs = sorted(conf_energy_pairs, key=lambda x: x[1])
    # Take top 'keep' by energy but ensure diversity with RMSD pruning
    ranked = [cid for cid,_ in conf_energy_pairs]
    diverse = prune_by_rmsd(mol, ranked, rms_cut=RMS_PRUNE)
    # keep the best among those diverse; if too many, cap at 'keep'
    diverse_sorted = sorted(diverse, key=lambda cid: dict(conf_energy_pairs)[cid])
    return diverse_sorted[:keep]

def compute_ani_energies(mol, conf_ids, model_name='ani2x'):
    import torch, torchani
    # Load model
    model = (torchani.models.ANI2x() if model_name.lower() == 'ani2x'
             else torchani.models.ANI1ccx())
    device = torch.device('cpu')
    model = model.to(device).eval()

    # Map atomic numbers -> element symbols for TorchANI
    z2sym = {1:'H', 6:'C', 7:'N', 8:'O', 9:'F', 16:'S', 17:'Cl', 35:'Br', 53:'I'}
    symbols = [z2sym[atom.GetAtomicNum()] for atom in mol.GetAtoms()]

    # TorchANI helper to build species tensor
    species = model.consts.species_to_tensor(symbols).unsqueeze(0).to(device)  # shape (1, natoms)

    energies = {}
    for cid in conf_ids:
        conf = mol.GetConformer(cid)
        coords = [[conf.GetAtomPosition(i).x,
                   conf.GetAtomPosition(i).y,
                   conf.GetAtomPosition(i).z] for i in range(mol.GetNumAtoms())]
        coordinates = torch.tensor([coords], dtype=torch.float32, device=device)  # (1, natoms, 3)
        with torch.no_grad():
            e = model((species, coordinates)).energies.item()  # Hartree
        energies[cid] = e
    return energies  # dict: confId -> Eh


def write_sdf(mol, conf_ids, fields, path):
    w = Chem.SDWriter(path)
    for cid in conf_ids:
        m = Chem.Mol(mol)
        m.SetProp("_Name", f"{NAME}_conf{cid}")
        for k,v in fields.items():
            if cid in v:
                m.SetDoubleProp(k, float(v[cid]))
        w.write(m, confId=cid)
    w.close()

# === 1) Prepare input ===
mol0 = prepare_mol(SMILES)

# (Optional) you’d generate protomers externally (e.g., Dimorphite at pH 7) and loop each through the same steps.
tauts = [mol0]

print(f"Found {len(tauts)} unique tautomers")
best_overall = None  # (energy_Eh, taut_idx, conf_id)

for i, taut in enumerate(tauts):
    confEs = embed_minimize_confs(taut, NCONF)
    keep_ids = mmff_rank_and_prune(taut, confEs, keep=KEEP_MMFF)

    aniE = {}
    if USE_ANI:
        aniE = compute_ani_energies(taut, keep_ids, model_name='ani2x')

    # Report for this tautomer
    if USE_ANI and aniE:
        cid_min = min(aniE, key=lambda k: aniE[k])
        E_min = aniE[cid_min]   # Hartree
        print(f"[taut {i}] lowest ANI energy: {E_min:.9f} Eh  (conf {cid_min})")
        if (best_overall is None) or (E_min < best_overall[0]):
            best_overall = (E_min, i, cid_min)
    else:
        # fall back to MMFF (NOT electronic) just so something prints
        mmffE = dict(confEs)
        cid_min = min(keep_ids, key=lambda k: mmffE[k])
        print(f"[taut {i}] lowest MMFF (not electronic): {mmffE[cid_min]:.3f} kcal/mol  (conf {cid_min})")

if USE_ANI and best_overall:
    E, ti, ci = best_overall
    print(f"\nGround-state estimate (ANI): {E:.9f} Eh  from tautomer {ti}, conformer {ci}")


Found 1 unique tautomers


ModuleNotFoundError: No module named 'distutils'

In [20]:
!pip install rdkit
!pip install torch
!pip install torchani
!pip install distutils


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement distutils (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for distutils


In [8]:
from rdkit.Chem import AllChem, rdMolAlign, MolStandardize

In [23]:
!pip install setuptools setuptools-scm packaging
!pip install setuptools-distutils



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement setuptools-distutils (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for setuptools-distutils


# attempt 2


In [28]:
import os, shutil, subprocess, sys, math, tempfile

# ---- Config ----
SMILES = "C[C@@H](N)C(=O)O"   # L-alanine (PubChem CID 5950)
CHARGE = 0
MULTIPLICITY = 1              # 2S+1; 1 for closed-shell
REF_ENERGY_HA = -248.3843246  # BVWN/TZV(2p,d) ref (gas phase); see citation in the writeup

# ---- RDKit bits: SMILES -> 3D geometry ----
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
except ImportError:
    print("Please install RDKit (e.g., conda install -c conda-forge rdkit)")
    sys.exit(1)

def smiles_to_3d_xyz(smiles, name="molecule"):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES")
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    if AllChem.EmbedMolecule(mol, params) != 0:
        raise RuntimeError("ETKDG embedding failed")
    AllChem.MMFFOptimizeMolecule(mol, maxIters=200)
    conf = mol.GetConformer()
    symbols = [a.GetSymbol() for a in mol.GetAtoms()]
    coords = [conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())]
    lines = [str(len(symbols)), name]
    for s, p in zip(symbols, coords):
        lines.append(f"{s:2s} {p.x: .8f} {p.y: .8f} {p.z: .8f}")
    return "\n".join(lines) + "\n"

def run_xtb_single_point(xyz_path, charge=0, multiplicity=1, gfn=2):
    xtb = shutil.which("xtb")
    if not xtb:
        raise RuntimeError("xtb binary not found on PATH. Install xtb and ensure 'xtb' is available.")
    # multiplicity u = 2S (so singlet -> u=0, doublet -> u=1, etc.)
    u = multiplicity - 1
    cmd = [xtb, xyz_path, "--gfn", str(gfn), "--sp", "-c", str(charge), "-u", str(u)]
    # Clean run directory
    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = env.get("OMP_NUM_THREADS", "1")
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    out = res.stdout
    # Parse "TOTAL ENERGY" line (Ha)
    E_ha = None
    for line in out.splitlines():
        if "TOTAL ENERGY" in line.upper() and "Eh" in line:
            # Example: "TOTAL ENERGY       -400.1234567 Eh"
            try:
                tokens = line.replace("Eh","").split()
                E_ha = float(tokens[-1])
            except Exception:
                pass
    if E_ha is None:
        # Fallback: look for "total energy" with different casing
        for line in out.splitlines():
            if "total energy" in line.lower() and "Eh" in line:
                try:
                    tokens = line.replace("Eh","").split()
                    E_ha = float(tokens[-1])
                except Exception:
                    pass
    if E_ha is None:
        print(out)
        raise RuntimeError("Could not parse total energy from xTB output above.")
    return E_ha, out

def ha_to_kjmol(E_ha):
    return E_ha * 2625.49962

def main():
    with tempfile.TemporaryDirectory() as td:
        xyz = smiles_to_3d_xyz(SMILES, name="alanine")
        xyz_path = os.path.join(td, "alanine.xyz")
        with open(xyz_path, "w") as f:
            f.write(xyz)
        E_ha, raw = run_xtb_single_point(xyz_path, CHARGE, MULTIPLICITY, gfn=2)
        E_kj = ha_to_kjmol(E_ha)

        print("=== Alanine (GFN2-xTB single-point) ===")
        print(f"SMILES: {SMILES}")
        print(f"Charge/Multiplicity: {CHARGE}/{MULTIPLICITY}")
        print(f"Total energy: {E_ha:.9f} Ha  ({E_kj:,.2f} kJ/mol)")

        if REF_ENERGY_HA is not None:
            diff_ha = E_ha - REF_ENERGY_HA
            diff_kj = ha_to_kjmol(diff_ha)
            print("\n--- Comparison vs reference ---")
            print(f"Reference (literature) energy: {REF_ENERGY_HA:.9f} Ha")
            print(f"Difference: {diff_ha:+.9f} Ha  ({diff_kj:+,.2f} kJ/mol)")
            print("(Note: references depend strongly on method/basis/geometry; differences of many Hartree across methods are normal.)")

if __name__ == "__main__":
    main()


      -----------------------------------------------------------      
     |                   =====================                   |     
     |                           x T B                           |     
     |                   =====================                   |     
     |                         S. Grimme                         |     
     |          Mulliken Center for Theoretical Chemistry        |     
     |                    University of Bonn                     |     
      -----------------------------------------------------------      

   * xtb version 6.7.1 (edcfbbe) compiled by 'runner@Mac-1756978095027.local' on 2025-09-04

   xtb is free software: you can redistribute it and/or modify it under
   the terms of the GNU Lesser General Public License as published by
   the Free Software Foundation, either version 3 of the License, or
   (at your option) any later version.
   
   xtb is distributed in the hope that it will be useful,
   but WITHOUT ANY

RuntimeError: Could not parse total energy from xTB output above.

In [ ]:
!pip install xtb


SyntaxError: invalid syntax (411378046.py, line 2)